<a href="https://colab.research.google.com/github/J-Aamir/SpotterAI_Assess/blob/main/Pyfiles/Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [81]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

In [82]:
# ==========================================
# 1. LOAD DATA DIRECTLY FROM GITHUB RAW URLS
# ==========================================
BASE_RAW_URL = "https://raw.githubusercontent.com/J-Aamir/SpotterAI_Assess/main/Datasets/"

train_url = BASE_RAW_URL + "train-test.csv"
val_url = BASE_RAW_URL + "validation.csv"
dec_url = BASE_RAW_URL + "december-chart-inputs.csv"

print("Downloading and loading datasets from GitHub...")
train_df = pd.read_csv(train_url)
val_df = pd.read_csv(val_url)
dec_df = pd.read_csv(dec_url)


In [83]:
# ==========================================
# 2. NEAT VARIABLE OVERVIEW & MISSING DATA ANALYSIS
# ==========================================
def inspect_dataset(df: pd.DataFrame, name: str) -> pd.DataFrame:
    overview = []
    for col in df.columns:
        col_type = df[col].dtype
        missing_count = df[col].isnull().sum()
        missing_pct = round((missing_count / len(df)) * 100, 2)
        unique_count = df[col].nunique()

        # Suggested imputation strategy based on variable nature
        if missing_count == 0:
            strategy = "No Missing Values"
        elif pd.api.types.is_numeric_dtype(df[col]):
            # Check for skewness to decide between Mean vs Median
            skewness = df[col].skew()
            strategy = "Median Imputation" if abs(skewness) > 0.5 else "Mean Imputation"
        else:
            strategy = "Mode (Most Frequent) Imputation"

        overview.append({
            "Column Name": col,
            "Data Type": col_type,
            "Missing Count": missing_count,
            "Missing (%)": missing_pct,
            "Unique Values": unique_count,
            "Dealing Strategy": strategy
        })

    summary_df = pd.DataFrame(overview)
    print(f"\n================ {name} Overview ================")
    print(f"Total Rows: {len(df):,}, Total Columns: {len(df.columns)}")
    return summary_df

# Display analysis for training data
overview_summary = inspect_dataset(train_df, "Train-Test Dataset")
print(overview_summary.to_string(index=False))



================ Train-Test Dataset Overview ================
Total Rows: 48,000, Total Columns: 14
 Column Name Data Type  Missing Count  Missing (%)  Unique Values  Dealing Strategy
     load_id    object              0         0.00          48000 No Missing Values
      pickup    object              0         0.00             64 No Missing Values
    delivery    object              0         0.00             64 No Missing Values
  pickup_lat   float64              0         0.00             64 No Missing Values
  pickup_lon   float64              0         0.00             63 No Missing Values
delivery_lat   float64              0         0.00             64 No Missing Values
delivery_lon   float64              0         0.00             63 No Missing Values
    distance   float64              0         0.00          21204 No Missing Values
   equipment    object              0         0.00              3 No Missing Values
      weight   float64            300         0.62         

In [84]:
# ==========================================
# 3. EXPLICIT IMPUTATION & FEATURE ENGINEERING
# ==========================================

# Step A: Compute imputation values strictly from TRAINING data to prevent data leakage
train_fill_values = {}

# Analyze numerical columns in training data
for col in train_df.select_dtypes(include=[np.number]).columns:
    if train_df[col].isnull().any():
        skewness = train_df[col].skew()
        if abs(skewness) > 0.5:
            train_fill_values[col] = train_df[col].median()
            print(f"Training set: Imputing {col} with Median (Skewness: {skewness:.2f})")
        else:
            train_fill_values[col] = train_df[col].mean()
            print(f"Training set: Imputing {col} with Mean (Skewness: {skewness:.2f})")

def clean_and_engineer_features(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    df = df.copy()

    # 1. Date Feature Engineering
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df['month'] = df['date'].dt.month
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['day_of_year'] = df['date'].dt.dayofyear
        df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
        df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

    # 2. Apply pre-computed training imputation values across all datasets safely
    for col, fill_val in train_fill_values.items():
        if col in df.columns and df[col].isnull().any():
            df[col] = df[col].fillna(fill_val)

    return df

# Apply feature engineering and safe filling
train_clean = clean_and_engineer_features(train_df, is_train=True)
val_clean = clean_and_engineer_features(val_df, is_train=False)
dec_clean = clean_and_engineer_features(dec_df, is_train=False)

# Define feature sets (now including day_of_year and is_month_end)
num_features = ['distance', 'weight', 'market_index', 'quote_signal',
                'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon',
                'month', 'day_of_week', 'day_of_month', 'day_of_year',
                'is_weekend', 'is_month_end']
cat_features = ['equipment', 'pickup', 'delivery']
target = 'posted_rate'

# Validate post-imputation missing count
print("\nRemaining missing values in clean training dataset:")
print(train_clean[num_features + cat_features].isnull().sum())

Training set: Imputing weight with Median (Skewness: -1.91)
Training set: Imputing market_index with Mean (Skewness: 0.21)

Remaining missing values in clean training dataset:
distance        0
weight          0
market_index    0
quote_signal    0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
month           0
day_of_week     0
day_of_month    0
day_of_year     0
is_weekend      0
is_month_end    0
equipment       0
pickup          0
delivery        0
dtype: int64


In [85]:
# ==========================================
# 4. LOCAL TRAIN / TEST SPLIT FOR EVALUATION
# ==========================================
X = train_clean[num_features + cat_features]
y = train_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

def print_metrics(model_name: str, y_true: pd.Series, y_pred: np.ndarray) -> float:
    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"--- {model_name} Evaluation ---")
    print(f"  RMSE : ${rmse:.2f}")
    print(f"  R²   : {r2:.4f}\n")
    return rmse

In [86]:
# ==========================================
# 5. MODEL 1: LINEAR REGRESSION PIPELINE
# ==========================================
lr_num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

lr_cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

lr_preprocessor = ColumnTransformer(transformers=[
    ('num', lr_num_transformer, num_features),
    ('cat', lr_cat_transformer, cat_features)
])

linear_model = Pipeline([
    ('preprocessor', lr_preprocessor),
    ('regressor', LinearRegression())
])

print("\nTraining Linear Regression Baseline...")
linear_model.fit(X_train, y_train)
lr_preds = linear_model.predict(X_test)
lr_rmse = print_metrics("Linear Regression Baseline", y_test, lr_preds)




Training Linear Regression Baseline...
--- Linear Regression Baseline Evaluation ---
  RMSE : $536.33
  R²   : 0.8655



In [87]:
# ==========================================
# 6. MODEL 2: HIST GRADIENT BOOSTING & ENSEMBLE
# ==========================================
hgb_cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

hgb_preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', hgb_cat_transformer, cat_features)
])

hgb_model = Pipeline([
    ('preprocessor', hgb_preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42
    ))
])

print("Training HistGradientBoosting Regressor...")
hgb_model.fit(X_train, y_train)
hgb_preds = hgb_model.predict(X_test)
hgb_rmse = print_metrics("HistGradientBoosting Regressor", y_test, hgb_preds)

# Combine HGB and LinearRegression into a Voting Ensemble
ensemble_model = VotingRegressor(
    estimators=[('hgb', hgb_model), ('lr', linear_model)],
    weights=[0.8, 0.2]
)

print("Training Weighted Voting Ensemble...")
ensemble_model.fit(X_train, y_train)
ens_preds = ensemble_model.predict(X_test)
ens_rmse = print_metrics("Weighted Voting Ensemble", y_test, ens_preds)

Training HistGradientBoosting Regressor...
--- HistGradientBoosting Regressor Evaluation ---
  RMSE : $533.79
  R²   : 0.8668

Training Weighted Voting Ensemble...
--- Weighted Voting Ensemble Evaluation ---
  RMSE : $530.91
  R²   : 0.8682



In [88]:
# ==========================================
# 7. COMPARE MODELS & SELECT THE WINNER
# ==========================================

# Select the Ensemble as the winning deployment model
best_model = ensemble_model
best_name = "Weighted Voting Ensemble"

print(f"\n🏆 Winner Selected: {best_name}")

# Retrain the winning model on the full 48,000 training dataset
print(f"Retraining {best_name} on the full 48,000 training dataset...")
best_model.fit(X, y)

print("\nModel training and selection complete!")


🏆 Winner Selected: Weighted Voting Ensemble
Retraining Weighted Voting Ensemble on the full 48,000 training dataset...

Model training and selection complete!


In [89]:
# ==========================================
# 8. RUN WINNING MODEL ON VALIDATION DATASET & EXPORT
# ==========================================

# Check missing values in validation.csv across ALL columns before cleaning
print("Missing values across all columns before cleaning:")
print(val_df.isnull().sum()[val_df.isnull().sum() > 0])

# Apply cleaning function (which applies training mean/median strategies safely)
val_clean = clean_and_engineer_features(val_df)

# Check missing values across ALL feature columns after cleaning (Should be all 0)
print("\nMissing values across all features after cleaning (Should be all 0):")
print(val_clean[num_features + cat_features].isnull().sum()[val_clean[num_features + cat_features].isnull().sum() > 0])

print("\nPreparing validation dataset for inference...")
# val_clean is generated from validation.csv using our leakage-free feature engineering function
X_val = val_clean[num_features + cat_features]

print("Generating predictions on validation.csv using the winning model...")
val_preds = np.clip(best_model.predict(X_val), a_min=1.0, a_max=None)

# Load the official template to guarantee correct row count and load_id mapping
template_url = BASE_RAW_URL + "validation-predictions-template.csv"
template_df = pd.read_csv(template_url)

# Populate the template's predicted_rate column securely mapping by load_id
val_predictions_map = dict(zip(val_clean['load_id'], val_preds))
template_df['predicted_rate'] = template_df['load_id'].map(val_predictions_map)

# Verify that all 12,000 rows were successfully filled with no missing values
missing_preds = template_df['predicted_rate'].isnull().sum()
print(f"Missing predictions check: {missing_preds}")

# Save precisely as required by the assessment instructions
template_df.to_csv("validation_predictions.csv", index=False)
print("Successfully saved: validation_predictions.csv")

Missing values across all columns before cleaning:
weight          165
market_index    249
dtype: int64

Missing values across all features after cleaning (Should be all 0):
Series([], dtype: int64)

Preparing validation dataset for inference...
Generating predictions on validation.csv using the winning model...
Missing predictions check: 0
Successfully saved: validation_predictions.csv


In [90]:
# ==========================================
# 9. GENERATE DECEMBER CHART PREDICTIONS
# ==========================================

print("Preparing December dataset for inference...")
dec_clean = dec_df.copy()

# 1. Temporal feature engineering from date
dec_clean['date'] = pd.to_datetime(dec_clean['date'])
dec_clean['month'] = dec_clean['date'].dt.month
dec_clean['day_of_week'] = dec_clean['date'].dt.dayofweek
dec_clean['day_of_month'] = dec_clean['date'].dt.day
dec_clean['day_of_year'] = dec_clean['date'].dt.dayofyear
dec_clean['is_weekend'] = dec_clean['day_of_week'].isin([5, 6]).astype(int)
dec_clean['is_month_end'] = dec_clean['date'].dt.is_month_end.astype(int)

# 2. Impute/fill market indicators and coordinates using training statistics
dec_clean['market_index'] = train_df['market_index'].mean()
dec_clean['quote_signal'] = train_df['quote_signal'].median()
dec_clean['pickup_lat'] = train_df[train_df['pickup'] == 'Lexington']['pickup_lat'].mean()
dec_clean['pickup_lon'] = train_df[train_df['pickup'] == 'Lexington']['pickup_lon'].mean()
dec_clean['delivery_lat'] = train_df[train_df['delivery'] == 'Fort Wayne']['delivery_lat'].mean()
dec_clean['delivery_lon'] = train_df[train_df['delivery'] == 'Fort Wayne']['delivery_lon'].mean()
dec_clean['weight'] = dec_clean['weight'].fillna(train_df['weight'].median())

# 3. Predict rates using your winning model
X_dec = dec_clean[num_features + cat_features]
dec_preds = np.clip(best_model.predict(X_dec), a_min=1.0, a_max=None)

# 4. Format and save precisely matching the 7 required columns
december_export = dec_df[['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date']].copy()
december_export['date'] = pd.to_datetime(december_export['date']).dt.strftime('%Y-%m-%d')
december_export['predicted_rate'] = dec_preds
december_export = december_export[["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]]

december_export.to_csv("december-chart-inputs.csv", index=False)
print("Successfully generated predictions for december-chart-inputs.csv!")
print(f"Unique predicted rates across December: {december_export['predicted_rate'].nunique()}")

Preparing December dataset for inference...
Successfully generated predictions for december-chart-inputs.csv!
Unique predicted rates across December: 31


In [91]:
import urllib.request
import os

# 1. Download official score.py directly into Colab to ensure it's present
score_url = "https://raw.githubusercontent.com/J-Aamir/SpotterAI_Assess/main/Pyfiles/score.py"
try:
    urllib.request.urlretrieve(score_url, "score.py")
    print("score.py downloaded successfully!")
except Exception as e:
    print("Download note (if already present locally):", e)

# 2. Check that all required files are here
required_files = ['score.py', 'validation_predictions.csv', 'december-chart-inputs.csv']
missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print(f"⚠️ Missing files: {missing}. Make sure they are uploaded to /content/")
else:
    print("✅ All required files are present!")
    # 3. Run the validator and generate the chart
    !python score.py --predictions validation_predictions.csv --december-predictions december-chart-inputs.csv

score.py downloaded successfully!
✅ All required files are present!
Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.
